In [1]:
import pandas as pd
import re
from soynlp.normalizer import repeat_normalize

# --- [1. 정제 함수 정의] ---
def clean_text(text):
    """
    한글, 영문, 숫자 및 필수 문장부호만 남기고 정제하는 함수
    """
    if not isinstance(text, str):
        return ""
    
    # 1. 특수문자 제거: 한글, 영문, 숫자, 주요 문장부호(. , ? !)만 남김
    # 감정 분석에 방해가 되는 이모티콘이나 불필요한 기호를 지웁니다.
    text = re.sub(r'[^가-힣a-zA-Z0-9.,?!\s]', '', text)
    
    # 2. 반복 문자 정제: "ㅋㅋㅋㅋㅋ" -> "ㅋㅋ", "ㅠㅠㅠㅠ" -> "ㅠㅠ"
    # 과도한 반복을 축약하여 모델의 학습 효율을 높입니다.
    text = repeat_normalize(text, num_repeats=2)
    
    # 3. 공백 정제: 문장 앞뒤 공백 제거 및 연속된 공백 통합
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# --- [2. 각 데이터셋 정제 프로세스] ---

def process_and_save(input_filename, output_filename):
    print(f"진행 중: {input_filename} 데이터 정제 중...")
    try:
        # 데이터 로드
        df = pd.read_csv(input_filename)
        
        # 'utterance' 컬럼 정제 (함수 적용)
        df['utterance'] = df['utterance'].apply(clean_text)
        
        # 정제 후 발생한 빈 문장 및 결측치 제거
        df = df[df['utterance'].str.strip() != ""]
        df = df.dropna(subset=['utterance'])
        
        # 정제 후 중복된 문장 제거
        df = df.drop_duplicates(subset=['utterance'])
        
        # 결과 저장 (한글 깨짐 방지 utf-8-sig)
        df.to_csv(output_filename, index=False, encoding='utf-8-sig')
        print(f"✅ 완료: {output_filename} 저장 완료 (총 {len(df)}건)")
        return df
    except FileNotFoundError:
        print(f"❌ 오류: {input_filename} 파일을 찾을 수 없습니다.")
    except Exception as e:
        print(f"❌ {input_filename} 처리 중 오류 발생: {e}")

# --- [3. 메인 실행] ---
if __name__ == "__main__":
    # 웰니스 데이터 정제
    process_and_save('pre_wellness.csv', 'cleaned_wellness.csv')
    
    # 챗봇 데이터 정제
    process_and_save('pre_chatbot.csv', 'cleaned_chatbot.csv')
    
    # 주제별 일상 대화 데이터 정제
    process_and_save('pre_subject_daily.csv', 'cleaned_subject.csv')

    print("\n✨ 모든 파일의 정제 과정이 끝났습니다.")

진행 중: pre_wellness.csv 데이터 정제 중...
✅ 완료: cleaned_wellness.csv 저장 완료 (총 19666건)
진행 중: pre_chatbot.csv 데이터 정제 중...
✅ 완료: cleaned_chatbot.csv 저장 완료 (총 5261건)
진행 중: pre_subject_daily.csv 데이터 정제 중...
✅ 완료: cleaned_subject.csv 저장 완료 (총 1370984건)

✨ 모든 파일의 정제 과정이 끝났습니다.


In [4]:
import pandas as pd
import re
from soynlp.normalizer import repeat_normalize

# 1. 이전과 동일한 정제 함수
def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[^가-힣a-zA-Z0-9.,?!\s]', '', text)
    text = repeat_normalize(text, num_repeats=2)
    return re.sub(r'\s+', ' ', text).strip()

# 2. 데이터 로드 및 정제 텍스트 생성
pre = pd.read_csv('pre_wellness.csv')
pre['cleaned_text'] = pre['utterance'].apply(clean_text)

# 3. 중복으로 판정되어 '버려질' 데이터들 (index 보존)
duplicates = pre[pre.duplicated(subset=['cleaned_text'], keep='first')].copy()

# 4. 버려진 각 데이터에 대해, '먼저 나타나서 자리를 차지한' 원본 찾기
comparison_list = []

for idx, row in duplicates.iterrows():
    # 현재 중복된 행과 'cleaned_text'가 똑같은 '첫 번째' 행을 찾습니다.
    original_row = pre[pre['cleaned_text'] == row['cleaned_text']].iloc[0]
    
    comparison_list.append({
        '삭제된 행 번호': idx,
        '삭제된 텍스트': row['utterance'],
        '살아남은 행 번호': original_row.name,
        '살아남은 텍스트': original_row['utterance'],
        '정제 후 동일해진 문구': row['cleaned_text']
    })

# 결과 출력
comparison_df = pd.DataFrame(comparison_list)
print("--- [중복 원인 분석 리포트] ---")
print(comparison_df[['삭제된 행 번호', '삭제된 텍스트', '살아남은 행 번호', '살아남은 텍스트']])

--- [중복 원인 분석 리포트] ---
    삭제된 행 번호                            삭제된 텍스트  살아남은 행 번호  \
0       1397                  그때부터 우울했던 것 같아요.        1015   
1       6245            뭔가 하고 싶은 의욕이 하나도 안 생기고…       6243   
2      13084          나도 힘든데 욕까지 먹으니까 너무 짜증났어요.      12961   
3      18004                 진짜 세상이 무너지는 것 같더라.      17990   
4      18521                         그냥 죽고 싶어요.      18394   
5      18630                        우울해서 죽고 싶어.       1508   
6      18631  그래서 너무 우울할 때면 막 죽고 싶은 생각이 종종 들었어.       1509   
7      18632                 우울하고, 멍청하고, 죽고 싶고.       1510   
8      18633                진짜… 너무 우울해서 죽고만 싶어.       1511   
9      18635              너무 우울해서 그냥 죽어버리고 싶어요…       1512   
10     19609                         이유 없이 불안해…      19267   

                             살아남은 텍스트  
0                    그때부터 우울했던 것 같아요.  
1            뭔가 하고 싶은 의욕이 하나도 안 생기고…   
2          나도 힘든데 욕까지 먹으니까 너무 짜증났어요.   
3                 진짜… 세상이 무너지는 것 같더라.  
4                         그냥… 죽고